# 1.0. Análisis exploratorio para la selección del periodo histórico

## Objetivo

Los datos históricos de MiBici abarcan varios años de operación. Sin embargo,
utilizar toda la información disponible no necesariamente implica utilizar los
datos más representativos para el desarrollo de modelos de Machine Learning.

A lo largo del tiempo, el sistema MiBici ha experimentado cambios en su nivel
de demanda y en su infraestructura, incluyendo modificaciones en el número de
estaciones disponibles.

El objetivo de este análisis exploratorio es evaluar la calidad, continuidad y
comportamiento temporal de los datos para determinar una ventana histórica
adecuada para el desarrollo de los modelos.

En particular, se analizarán:

1. Disponibilidad y continuidad de los datos históricos.
2. Consistencia de los archivos.
3. Evolución del número de viajes.
4. Evolución del número de estaciones activas.
5. Posibles cambios estructurales en el sistema.
6. Existencia de patrones estacionales.
7. Representatividad de distintas ventanas históricas.

La selección final buscará un equilibrio entre recencia, cantidad de información
y capacidad para capturar patrones estacionales.

## Importación de librerías

Se importan las librerías necesarias para localizar y procesar los archivos,
manipular los datos y generar las visualizaciones del análisis exploratorio.

- `pathlib` permite administrar rutas y archivos de manera independiente del
  sistema de archivos.
- `re` se utilizará para extraer el año y mes de los nombres de los archivos.
- `pandas` se utilizará para la lectura y procesamiento de los datos.
- `matplotlib` se utilizará para generar las visualizaciones.

In [ ]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt

## Identificación de los archivos disponibles

Como primer paso se revisa el repositorio de datos almacenado en Google Drive
para identificar cuántos archivos CSV se encuentran disponibles.

La carpeta contiene tanto los archivos históricos mensuales de viajes como un
catálogo de estaciones. En esta etapa únicamente se realiza un inventario de
todos los archivos CSV disponibles.

Esta revisión permite verificar que la fuente de datos sea accesible desde el
entorno de trabajo y conocer la cantidad total de archivos disponibles antes
de iniciar su procesamiento.

In [ ]:
RUTA_DATOS = Path(
    r"G:\Mi unidad\Colab Notebooks\Maestría\4.- Trimestre\07 - MLOps"
    r"\Semana 5 - Proyecto MiBici - Equipo 25\Bases de Datos - MiBici"
)

archivos_csv = sorted(
    RUTA_DATOS.glob("*.csv")
)

print(f"Ruta de datos: {RUTA_DATOS}")
print(f"Archivos CSV encontrados: {len(archivos_csv)}")

## Clasificación de las fuentes de datos

Los 143 archivos encontrados no corresponden exclusivamente a registros de
viajes.

Se identifican dos tipos de fuentes:

- **142 archivos mensuales de viajes**, cuyos nombres siguen el patrón
  `datos_abiertos_AAAA_MM.csv`.
- **1 catálogo de estaciones**, denominado `nomenclatura_2026_09.csv`.

El catálogo contiene información descriptiva de las estaciones y presenta una
estructura diferente a los archivos de viajes. Por esta razón, ambas fuentes
se separan antes de iniciar el procesamiento.

Esta separación evita que el catálogo sea procesado accidentalmente como un
archivo mensual de viajes.

In [ ]:
ARCHIVO_CATALOGO = (
    RUTA_DATOS / "nomenclatura_2026_09.csv"
)

archivos_viajes = sorted(
    RUTA_DATOS.glob("datos_abiertos_*.csv")
)

print(f"Archivos CSV totales: {len(archivos_csv)}")
print(f"Archivos mensuales de viajes: {len(archivos_viajes)}")
print(
    "Catálogo de estaciones encontrado:",
    ARCHIVO_CATALOGO.exists()
)

## Validación de las fuentes identificadas

Antes de procesar la información se verifica que el catálogo de estaciones
exista y que se hayan identificado archivos mensuales de viajes.

También se muestran el primer y último archivo de viajes encontrados para
comprobar visualmente la cobertura temporal disponible.

Estas validaciones permiten detectar de manera temprana problemas como archivos
faltantes, cambios en los nombres o errores en la ruta de datos.

In [ ]:
if not ARCHIVO_CATALOGO.exists():
    raise FileNotFoundError(
        f"No se encontró el catálogo: {ARCHIVO_CATALOGO.name}"
    )

if len(archivos_viajes) == 0:
    raise FileNotFoundError(
        "No se encontraron archivos mensuales de viajes."
    )

print("Validación completada correctamente.")

print(
    "Primer archivo de viajes:",
    archivos_viajes[0].name
)

print(
    "Último archivo de viajes:",
    archivos_viajes[-1].name
)

print(
    "Catálogo de estaciones:",
    ARCHIVO_CATALOGO.name
)

## Definición del esquema estándar de viajes

Los archivos mensuales contienen ocho variables asociadas con la identificación
del viaje, usuario, características demográficas, fechas y estaciones de origen
y destino.

Durante la exploración inicial se identificaron algunas inconsistencias en los
encabezados de ciertos archivos. Para garantizar que todos los meses sean
procesados bajo el mismo esquema, se define un conjunto estándar de nombres
de variables.

Esta normalización permitirá procesar los archivos de manera uniforme,
independientemente de pequeñas diferencias en sus encabezados originales.

In [ ]:
COLUMNAS_ESTANDAR = [
    "viaje_id",
    "usuario_id",
    "genero",
    "anio_nacimiento",
    "inicio_viaje",
    "fin_viaje",
    "origen_id",
    "destino_id"
]

## Lectura de archivos y tratamiento de codificaciones

Los archivos históricos no necesariamente fueron generados utilizando la misma
codificación de caracteres.

Durante la exploración inicial se identificaron archivos que pueden leerse
utilizando UTF-8 y otros que requieren Windows-1252 (`cp1252`).

Para evitar correcciones manuales, se implementa una función de lectura que
intenta primero UTF-8 y, en caso de encontrar un error de codificación,
intenta nuevamente utilizando CP1252.

La función también devuelve la codificación utilizada, lo que permitirá
documentar posteriormente la heterogeneidad encontrada en el histórico.

In [ ]:
def leer_csv_mibici(archivo, nrows=None):
    """
    Lee un archivo CSV utilizando las codificaciones
    identificadas en el histórico de MiBici.
    """

    codificaciones = [
        "utf-8",
        "cp1252"
    ]

    for encoding in codificaciones:

        try:

            df = pd.read_csv(
                archivo,
                encoding=encoding,
                nrows=nrows
            )

            return df, encoding

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"No fue posible leer el archivo {archivo.name}"
    )

## Normalización del esquema de los archivos de viajes

Para garantizar que todos los archivos mensuales tengan el mismo esquema,
se valida primero que cada archivo contenga ocho columnas.

Posteriormente, los encabezados se sustituyen por los nombres estándar
definidos previamente.

Esta estrategia permite corregir de manera reproducible pequeñas inconsistencias
en los encabezados sin modificar manualmente los archivos originales.

In [ ]:
def normalizar_columnas(df):
    """
    Normaliza los nombres de las columnas de viajes
    utilizando el esquema estándar.
    """

    if len(df.columns) != len(COLUMNAS_ESTANDAR):

        raise ValueError(
            f"Se esperaban {len(COLUMNAS_ESTANDAR)} columnas "
            f"y se encontraron {len(df.columns)}."
        )

    df = df.copy()

    df.columns = COLUMNAS_ESTANDAR

    return df

## Identificación del periodo esperado

Los archivos mensuales siguen una nomenclatura que incluye explícitamente
el año y mes al que corresponden.

Por ejemplo:

`datos_abiertos_2024_09.csv`

corresponde a septiembre de 2024.

Se utiliza esta información para obtener el periodo esperado de cada archivo.
Además de ordenar cronológicamente los datos, esta información servirá como
control de calidad para validar posteriormente la interpretación de las fechas.

In [ ]:
def obtener_periodo_archivo(archivo):
    """
    Extrae el año y mes del nombre del archivo.

    Ejemplo:
    datos_abiertos_2024_09.csv -> 2024, 9
    """

    match = re.search(
        r"(\d{4})_(\d{2})",
        archivo.name
    )

    if not match:

        raise ValueError(
            f"No se pudo identificar el periodo "
            f"de {archivo.name}"
        )

    anio = int(match.group(1))
    mes = int(match.group(2))

    return anio, mes

## Normalización de fechas

Durante la exploración inicial se identificó que los archivos históricos no
utilizan de manera completamente uniforme la representación de las fechas.

Una fecha como `01/11/2023`, por ejemplo, puede ser interpretada como:

- 11 de enero de 2023; o
- 1 de noviembre de 2023.

Para evitar imponer manualmente un único formato, se prueban dos estrategias:

1. Interpretación estándar de `pandas`.
2. Interpretación considerando primero el día (`dayfirst=True`).

El año y mes contenidos en el nombre del archivo se utilizan como referencia.
Se selecciona automáticamente la estrategia que genera el mayor número de
fechas dentro del periodo esperado.

De esta forma, la normalización de fechas se realiza mediante una regla
reproducible y no mediante correcciones manuales por archivo.

In [ ]:
def convertir_fechas_mibici(serie, archivo):

    anio_esperado, mes_esperado = (
        obtener_periodo_archivo(archivo)
    )

    opciones = {}

    for estrategia, dayfirst in [
        ("default", False),
        ("dayfirst", True)
    ]:

        fechas = pd.to_datetime(
            serie,
            errors="coerce",
            dayfirst=dayfirst
        )

        coincide_periodo = (
            (fechas.dt.year == anio_esperado) &
            (fechas.dt.month == mes_esperado)
        )

        opciones[estrategia] = {
            "fechas": fechas,
            "coincidencias": coincide_periodo.sum()
        }

    mejor_estrategia = max(
        opciones,
        key=lambda x: opciones[x]["coincidencias"]
    )

    return (
        opciones[mejor_estrategia]["fechas"],
        mejor_estrategia
    )

## Auditoría de los archivos históricos de viajes

Antes de realizar el análisis temporal se verifica que los 142 archivos
mensuales puedan procesarse utilizando las reglas de ingesta definidas.

Para cada archivo se revisa:

- si puede ser leído correctamente;
- qué codificación fue necesaria;
- cuántas columnas contiene;
- si puede normalizarse al esquema esperado.

En esta etapa únicamente se leen algunas filas de cada archivo, ya que el
objetivo es validar su estructura y no procesar todavía todos los registros.

In [ ]:
auditoria = []

for archivo in archivos_viajes:

    try:

        df, encoding = leer_csv_mibici(
            archivo,
            nrows=10
        )

        n_columnas_original = len(df.columns)

        df = normalizar_columnas(df)

        auditoria.append({
            "archivo": archivo.name,
            "encoding": encoding,
            "n_columnas": n_columnas_original,
            "estado": "OK"
        })

    except Exception as e:

        auditoria.append({
            "archivo": archivo.name,
            "encoding": None,
            "n_columnas": None,
            "estado": str(e)
        })


auditoria = pd.DataFrame(auditoria)

## Resultados de la auditoría

Una vez revisados los archivos, se resumen los resultados de la auditoría.

El objetivo es comprobar que todos los archivos puedan procesarse
automáticamente e identificar las diferentes codificaciones presentes en
el histórico.

In [ ]:
print(
    "Total de archivos de viajes:",
    len(auditoria)
)

print("\nEstado de lectura:")

print(
    auditoria["estado"].value_counts()
)

print("\nCodificaciones identificadas:")

print(
    auditoria["encoding"].value_counts()
)

### Hallazgo

Los 142 archivos mensuales pueden procesarse automáticamente mediante las
reglas de ingesta definidas.

Se identificaron dos codificaciones de caracteres:

- 99 archivos utilizan UTF-8.
- 43 archivos requieren Windows-1252 (CP1252).

Por lo tanto, la fuente histórica presenta heterogeneidad en su codificación,
la cual debe ser considerada por el pipeline de ingesta para garantizar un
procesamiento reproducible.

## Exploración del catálogo de estaciones

Además de los archivos históricos de viajes, se dispone del catálogo
`nomenclatura_2026_09.csv`.

Este archivo contiene información descriptiva y geográfica de las estaciones,
incluyendo:

- identificador;
- nombre;
- latitud;
- longitud;
- capacidad.

El catálogo podrá utilizarse posteriormente para enriquecer los registros de
viajes con características propias de las estaciones.

En esta etapa se realiza una revisión inicial de su estructura y calidad.

In [ ]:
catalogo_estaciones, encoding_catalogo = (
    leer_csv_mibici(
        ARCHIVO_CATALOGO
    )
)

print(
    "Codificación utilizada:",
    encoding_catalogo
)

print(
    "Número de registros:",
    len(catalogo_estaciones)
)

print(
    "Número de columnas:",
    len(catalogo_estaciones.columns)
)

display(
    catalogo_estaciones.head()
)

In [ ]:
catalogo_estaciones.info()

## Validación básica del catálogo de estaciones

Se revisa la integridad básica del catálogo para identificar valores faltantes,
identificadores duplicados y el número de estaciones disponibles.

El identificador `id` será particularmente importante, ya que permitirá
relacionar posteriormente el catálogo con las variables `origen_id` y
`destino_id` de los registros de viajes.

In [ ]:
print(
    "Número de estaciones:",
    catalogo_estaciones["id"].nunique()
)

print(
    "IDs duplicados:",
    catalogo_estaciones["id"].duplicated().sum()
)

print("\nValores faltantes:")

print(
    catalogo_estaciones.isna().sum()
)

In [ ]:
catalogo_estaciones.describe(
    include="all"
)

## Construcción del resumen histórico mensual

Una vez validada la estructura de los archivos, se procesa cada mes para
construir una tabla histórica agregada.

Para seleccionar el periodo de entrenamiento no es necesario concatenar
inicialmente todos los viajes del histórico. En su lugar, cada archivo se
procesa individualmente y se calculan indicadores mensuales.

Para cada periodo se obtienen:

- número total de viajes;
- número de usuarios únicos;
- número de estaciones activas;
- fecha mínima y máxima;
- número de fechas inválidas;
- número de registros fuera del periodo esperado;
- estrategia utilizada para interpretar las fechas.

Esta tabla será la base del análisis temporal.

In [ ]:
resumen = []

for i, archivo in enumerate(
    archivos_viajes,
    start=1
):

    print(
        f"\rProcesando {i}/{len(archivos_viajes)}",
        end=""
    )

    # Lectura
    df, encoding = leer_csv_mibici(
        archivo
    )

    # Normalización del esquema
    df = normalizar_columnas(
        df
    )

    # Periodo esperado
    anio, mes = obtener_periodo_archivo(
        archivo
    )

    # Normalización de fechas
    df["inicio_viaje"], estrategia_inicio = (
        convertir_fechas_mibici(
            df["inicio_viaje"],
            archivo
        )
    )

    df["fin_viaje"], estrategia_fin = (
        convertir_fechas_mibici(
            df["fin_viaje"],
            archivo
        )
    )

    # Estaciones con actividad
    estaciones = pd.concat([
        df["origen_id"],
        df["destino_id"]
    ])

    # Validación del periodo
    fuera_periodo = (
        (df["inicio_viaje"].dt.year != anio) |
        (df["inicio_viaje"].dt.month != mes)
    )

    resumen.append({

        "periodo":
            f"{anio}-{mes:02d}",

        "archivo":
            archivo.name,

        "encoding":
            encoding,

        "estrategia_fecha":
            estrategia_inicio,

        "numero_viajes":
            len(df),

        "usuarios_unicos":
            df["usuario_id"].nunique(),

        "estaciones_activas":
            estaciones.nunique(),

        "fecha_min":
            df["inicio_viaje"].min(),

        "fecha_max":
            df["inicio_viaje"].max(),

        "fechas_invalidas":
            df["inicio_viaje"].isna().sum(),

        "fechas_fuera_periodo":
            fuera_periodo.sum()
    })


resumen_mensual = pd.DataFrame(
    resumen
)

## Ordenamiento del resumen mensual

El periodo extraído de cada nombre de archivo se transforma a un tipo temporal
mensual (`Period`) y los registros se ordenan cronológicamente.

Esto permite realizar posteriormente operaciones temporales como identificar
meses faltantes, seleccionar ventanas históricas y comparar diferentes años.

In [ ]:
resumen_mensual["periodo"] = (
    pd.PeriodIndex(
        resumen_mensual["periodo"],
        freq="M"
    )
)

resumen_mensual = (
    resumen_mensual
    .sort_values("periodo")
    .reset_index(drop=True)
)

display(
    resumen_mensual.head()
)

display(
    resumen_mensual.tail()
)

## Validación de continuidad temporal

Antes de estudiar la evolución histórica se verifica que exista información
para todos los meses comprendidos entre el primer y último periodo disponible.

También se revisa que la normalización de fechas no haya producido fechas
inválidas o registros asignados a un mes diferente al esperado.

Estas validaciones son importantes porque cualquier interrupción o error
temporal podría afectar posteriormente la construcción de las particiones
de entrenamiento, validación y prueba.

In [ ]:
periodos_esperados = pd.period_range(
    resumen_mensual["periodo"].min(),
    resumen_mensual["periodo"].max(),
    freq="M"
)

periodos_faltantes = (
    periodos_esperados.difference(
        resumen_mensual["periodo"]
    )
)

print(
    "Periodo disponible:",
    resumen_mensual["periodo"].min(),
    "-",
    resumen_mensual["periodo"].max()
)

print(
    "Meses disponibles:",
    len(resumen_mensual)
)

print(
    "Meses esperados:",
    len(periodos_esperados)
)

print(
    "Meses faltantes:",
    len(periodos_faltantes)
)

print(
    "Fechas inválidas:",
    resumen_mensual["fechas_invalidas"].sum()
)

print(
    "Fechas fuera del periodo:",
    resumen_mensual["fechas_fuera_periodo"].sum()
)

print("\nEstrategias de fecha:")

print(
    resumen_mensual[
        "estrategia_fecha"
    ].value_counts()
)

### Hallazgo

Después de la normalización se obtiene una serie histórica continua desde
diciembre de 2014 hasta septiembre de 2026.

Los 142 meses esperados se encuentran disponibles, sin meses faltantes,
fechas inválidas ni registros asignados a periodos diferentes a los indicados
por sus archivos.

También se identificó que 3 archivos requieren una interpretación de fecha
con `dayfirst=True`, mientras que los 139 restantes utilizan la interpretación
estándar.

Estos resultados permiten continuar con el análisis temporal sobre una base
histórica completa y consistente.

## Evolución histórica del número de viajes

Una vez validada la calidad temporal de los datos, se analiza cómo ha
evolucionado el volumen mensual de viajes durante toda la historia disponible.

El objetivo es identificar tendencias, periodos de crecimiento, caídas
extraordinarias y posibles cambios estructurales que puedan afectar la
representatividad de los datos más antiguos para predecir la operación actual.

In [ ]:
x = (
    resumen_mensual["periodo"]
    .astype(str)
)

plt.figure(
    figsize=(14, 5)
)

plt.plot(
    x,
    resumen_mensual["numero_viajes"]
)

plt.xticks(
    range(
        0,
        len(x),
        12
    ),
    x.iloc[::12],
    rotation=45
)

plt.xlabel(
    "Periodo"
)

plt.ylabel(
    "Número de viajes"
)

plt.title(
    "Evolución mensual del número de viajes de MiBici"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

### Hallazgo

El volumen de viajes no presenta un comportamiento estable durante todo el
histórico.

Durante los primeros años se observa un crecimiento considerable de la
demanda. Posteriormente aparece una caída abrupta alrededor de 2020,
temporalmente compatible con el periodo de la pandemia de COVID-19.

Después de esta ruptura se observa una recuperación progresiva y un
comportamiento relativamente más estable durante los años recientes.

Esto sugiere que utilizar indiscriminadamente todo el histórico podría
incorporar patrones de demanda asociados con condiciones de operación
diferentes a las actuales.

## Evolución del número de estaciones activas

El volumen de viajes puede estar relacionado con el tamaño de la infraestructura
disponible. Por ello, además de analizar la demanda, se estudia la evolución
mensual del número de estaciones que registraron actividad como origen o destino.

Este análisis permite determinar si la configuración de la red se ha mantenido
estable o si existen etapas de expansión que puedan representar diferentes
regímenes operativos.

In [ ]:
plt.figure(
    figsize=(14, 5)
)

plt.plot(
    x,
    resumen_mensual["estaciones_activas"]
)

plt.xticks(
    range(
        0,
        len(x),
        12
    ),
    x.iloc[::12],
    rotation=45
)

plt.xlabel(
    "Periodo"
)

plt.ylabel(
    "Número de estaciones activas"
)

plt.title(
    "Evolución mensual de estaciones activas de MiBici"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

### Hallazgo

La infraestructura de MiBici ha cambiado significativamente durante el
periodo analizado.

El número de estaciones activas pasó de aproximadamente 86 al inicio del
histórico a 468 en septiembre de 2026, observándose distintas etapas de
expansión de la red.

Por lo tanto, los primeros años representan una configuración de infraestructura
considerablemente diferente a la operación reciente.

Este cambio estructural constituye un argumento para evitar utilizar
automáticamente toda la historia disponible para el entrenamiento.

## Viajes por estación activa

El incremento en el número total de viajes puede deberse tanto a un aumento de
la demanda como a la expansión del número de estaciones disponibles.

Para analizar ambos efectos se construye un indicador simple de intensidad de
uso, calculado como:

**Viajes por estación = Número de viajes / Número de estaciones activas**

Este indicador no representa la demanda individual de cada estación, sino una
medida agregada que permite comparar el volumen de viajes considerando el
tamaño de la red disponible en cada mes.

In [ ]:
resumen_mensual[
    "viajes_por_estacion"
] = (
    resumen_mensual["numero_viajes"] /
    resumen_mensual["estaciones_activas"]
)

## Evolución de los viajes por estación activa

Se analiza el indicador de viajes por estación durante todo el histórico para
determinar si el crecimiento del número total de viajes ha estado acompañado
por un incremento equivalente en la intensidad promedio de uso de la red.

In [ ]:
plt.figure(
    figsize=(14, 5)
)

plt.plot(
    x,
    resumen_mensual["viajes_por_estacion"]
)

plt.xticks(
    range(
        0,
        len(x),
        12
    ),
    x.iloc[::12],
    rotation=45
)

plt.xlabel(
    "Periodo"
)

plt.ylabel(
    "Viajes por estación activa"
)

plt.title(
    "Evolución mensual de viajes por estación activa"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

### Hallazgo

Al ajustar el número de viajes por la cantidad de estaciones activas se observa
que el crecimiento de los viajes totales no se traduce necesariamente en un
incremento equivalente en la intensidad promedio de uso de cada estación.

Durante los primeros años, el indicador de viajes por estación aumenta conforme
crece la demanda del sistema. Sin embargo, en los años recientes el indicador
presenta un comportamiento más estable, a pesar de que el número total de
estaciones continúa aumentando.

En particular, la expansión observada durante 2026 incrementa de manera
importante el número de estaciones disponibles y el volumen total de viajes,
pero no genera un crecimiento proporcional en los viajes por estación.

Esto sugiere que parte del incremento reciente en los viajes totales está
relacionado con la expansión de la infraestructura y no exclusivamente con un
aumento en la intensidad de uso de las estaciones existentes.

Por lo tanto, el número de estaciones activas representa un cambio estructural
relevante que debe considerarse al seleccionar el periodo histórico para el
modelado.

## Análisis de estacionalidad reciente

Además de los cambios estructurales, es importante determinar si existen
patrones de demanda que se repitan a lo largo del año.

Si existe un componente estacional anual, utilizar más de un año de información
puede ser conveniente para que los modelos observen varias veces los mismos
periodos estacionales.

Para este análisis se comparan los patrones mensuales de los años recientes,
a partir de 2022, evitando que las etapas iniciales de crecimiento de la red
dominen la comparación.

In [ ]:
resumen_mensual["anio"] = (
    resumen_mensual["periodo"].dt.year
)

resumen_mensual["mes"] = (
    resumen_mensual["periodo"].dt.month
)

recientes = resumen_mensual[
    resumen_mensual["anio"] >= 2022
].copy()

## Comparación del patrón mensual de viajes entre años

Se compara el número total de viajes de cada mes entre los años recientes.

El objetivo principal no es que los años tengan exactamente el mismo volumen,
sino identificar si existen meses que consistentemente presentan aumentos o
disminuciones de demanda.

La presencia de formas similares entre años proporcionaría evidencia de un
componente estacional.

In [ ]:
plt.figure(
    figsize=(12, 6)
)

for anio, datos in recientes.groupby("anio"):

    plt.plot(
        datos["mes"],
        datos["numero_viajes"],
        marker="o",
        label=str(anio)
    )

plt.xticks(
    range(1, 13),
    [
        "Ene", "Feb", "Mar", "Abr",
        "May", "Jun", "Jul", "Ago",
        "Sep", "Oct", "Nov", "Dic"
    ]
)

plt.xlabel(
    "Mes"
)

plt.ylabel(
    "Número de viajes"
)

plt.title(
    "Comportamiento mensual de viajes por año"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

## Comparación mensual de viajes por estación

Debido a que el número de estaciones ha cambiado con el tiempo, se repite el
análisis estacional utilizando el indicador de viajes por estación activa.

Esta comparación ayuda a distinguir entre aumentos del volumen total asociados
con la expansión de la red y cambios en la intensidad promedio de utilización
de las estaciones.

In [ ]:
plt.figure(
    figsize=(12, 6)
)

for anio, datos in recientes.groupby("anio"):

    plt.plot(
        datos["mes"],
        datos["viajes_por_estacion"],
        marker="o",
        label=str(anio)
    )

plt.xticks(
    range(1, 13),
    [
        "Ene", "Feb", "Mar", "Abr",
        "May", "Jun", "Jul", "Ago",
        "Sep", "Oct", "Nov", "Dic"
    ]
)

plt.xlabel(
    "Mes"
)

plt.ylabel(
    "Viajes por estación activa"
)

plt.title(
    "Comportamiento mensual de viajes por estación"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

## Comparación de ventanas históricas

Los análisis anteriores permiten identificar dos necesidades potencialmente
opuestas:

- utilizar información reciente para representar mejor la configuración actual
  del sistema;
- utilizar suficiente historia para capturar varios ciclos estacionales.

Para apoyar la selección del periodo se comparan cuatro ventanas históricas:

- 12 meses;
- 24 meses;
- 36 meses;
- 60 meses.

Para cada ventana se analiza el periodo cubierto, volumen de información y
rango de estaciones activas.

Esta comparación permitirá evaluar el equilibrio entre recencia, cantidad de
datos y representatividad de la infraestructura.

In [ ]:
ventanas = {
    "1 año": 12,
    "2 años": 24,
    "3 años": 36,
    "5 años": 60
}

comparacion = []

for nombre, n_meses in ventanas.items():

    datos = (
        resumen_mensual
        .tail(n_meses)
    )

    comparacion.append({

        "ventana":
            nombre,

        "desde":
            str(datos["periodo"].min()),

        "hasta":
            str(datos["periodo"].max()),

        "meses":
            len(datos),

        "numero_viajes":
            datos["numero_viajes"].sum(),

        "estaciones_min":
            datos["estaciones_activas"].min(),

        "estaciones_max":
            datos["estaciones_activas"].max(),

        "viajes_promedio_mes":
            datos["numero_viajes"].mean()
    })


comparacion_ventanas = (
    pd.DataFrame(comparacion)
)

display(
    comparacion_ventanas
)

## Ventana candidata de 36 meses

Como parte del análisis se evalúa una ventana correspondiente a los últimos
36 meses completos disponibles.

Considerando que el último periodo disponible es septiembre de 2026, esta
ventana comprende de octubre de 2023 a septiembre de 2026.

Una ventana de 36 meses permite disponer de tres ciclos anuales completos,
al mismo tiempo que concentra el análisis en una etapa considerablemente más
reciente de la operación de MiBici.

In [ ]:
ultimo_periodo = (
    resumen_mensual["periodo"].max()
)

inicio_ventana = (
    ultimo_periodo - 35
)

ventana_3_anios = resumen_mensual[
    resumen_mensual["periodo"] >= inicio_ventana
].copy()

print(
    "Inicio:",
    ventana_3_anios["periodo"].min()
)

print(
    "Fin:",
    ventana_3_anios["periodo"].max()
)

print(
    "Número de meses:",
    len(ventana_3_anios)
)

## Visualización de la ventana candidata

Finalmente, se representa la ventana de 36 meses sobre el histórico completo.

Esta visualización permite observar qué información histórica quedaría fuera
del periodo propuesto y evaluar visualmente si la ventana seleccionada se
concentra en un régimen operativo más reciente.

In [ ]:
plt.figure(
    figsize=(14, 5)
)

plt.plot(
    resumen_mensual["periodo"].astype(str),
    resumen_mensual["numero_viajes"],
    alpha=0.4,
    label="Histórico completo"
)

plt.plot(
    ventana_3_anios["periodo"].astype(str),
    ventana_3_anios["numero_viajes"],
    linewidth=2.5,
    label="Ventana candidata: 36 meses"
)

plt.xticks(
    range(
        0,
        len(resumen_mensual),
        12
    ),
    resumen_mensual[
        "periodo"
    ].astype(str).iloc[::12],
    rotation=45
)

plt.xlabel(
    "Periodo"
)

plt.ylabel(
    "Número de viajes"
)

plt.title(
    "Histórico disponible y ventana candidata para modelado"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

# Conclusiones del análisis exploratorio

El análisis exploratorio permitió evaluar la calidad, continuidad y
representatividad de la información histórica de MiBici antes de definir el
periodo que será utilizado para el desarrollo de los modelos.

En primer lugar, se comprobó que la fuente cuenta con información continua
desde diciembre de 2014 hasta septiembre de 2026, correspondiente a 142 meses
consecutivos. Después de aplicar las reglas de normalización desarrolladas no
se identificaron meses faltantes, fechas inválidas ni registros asignados a
periodos diferentes a los esperados.

Sin embargo, disponer de una serie histórica extensa no implica que toda la
información sea igualmente representativa de la operación actual.

El análisis del número de viajes muestra cambios importantes a lo largo del
tiempo. Durante los primeros años se observa una etapa de crecimiento, seguida
por una ruptura significativa alrededor de 2020 y una posterior recuperación.
Por otro lado, el análisis de las estaciones activas evidencia una expansión
considerable de la infraestructura, pasando de aproximadamente 86 estaciones
al inicio del histórico a 468 en septiembre de 2026.

Al analizar conjuntamente ambas variables mediante el indicador de viajes por
estación activa, se observa que el incremento del volumen total de viajes no
siempre está acompañado por un crecimiento proporcional en la intensidad
promedio de uso de las estaciones. Esto confirma que parte de los cambios
observados en la demanda agregada se encuentra relacionada con la expansión
de la red.

Adicionalmente, la comparación mensual de los años recientes muestra variaciones
intraanuales que justifican incorporar múltiples ciclos anuales dentro de la
muestra de entrenamiento.

## Selección del periodo

Considerando los hallazgos anteriores, se propone utilizar como ventana inicial
de modelado los últimos **36 meses completos**, comprendidos entre
**octubre de 2023 y septiembre de 2026**.

La selección de esta ventana busca equilibrar tres criterios:

1. **Recencia:** concentrar el entrenamiento en información más representativa
   de la operación actual de MiBici.

2. **Estacionalidad:** disponer de tres ciclos anuales completos que permitan
   capturar variaciones recurrentes de la demanda a lo largo del año.

3. **Volumen de información:** conservar una cantidad suficiente de
   observaciones para el entrenamiento y evaluación de los modelos.

Esta decisión también permite excluir etapas históricas asociadas con una
infraestructura considerablemente diferente y con eventos extraordinarios,
como la ruptura observada alrededor de 2020.

Por lo tanto, la ventana de **36 meses** se utilizará como periodo candidato
para las siguientes etapas del proyecto. Su conveniencia deberá ser validada
posteriormente mediante el desempeño fuera de muestra de los modelos,
comparándola, cuando sea necesario, con ventanas históricas alternativas.